In [5]:
#!/usr/bin/env python

"""
ETS baseline (statsforecast.AutoETS) for remaining-time prediction.

Assumptions:
- You already ran:
    1) warm-up trimming + train/val/test split
    2) prefix builder

So in `PREFIX_INPUT_FOLDER` you have files like:
    ..._warm10_train_prefix.csv
    ..._warm10_val_prefix.csv
    ..._warm10_test_prefix.csv

For each *base* log, we build TWO baselines:

1) Global ETS (single series over all cases)
   - Build a case-level cycle-time series from the TRAIN prefixes.
   - Fit AutoETS (Hyndman ETS) on that series.
   - Forecast horizon = (#val cases + #test cases).
   - Map forecasted total cycle times to val/test cases.
   - For every prefix row, compute:
        D_hat_ets  = predicted total cycle time for that case
        R_hat_ets  = max(D_hat_ets - elapsed_time, 0)

2) Per-variant ETS (one series per product variant),
   if a variant column is present:
   - For each variant:
       * Build a case-level cycle-time series for TRAIN cases of that variant.
       * Fit AutoETS on that series.
       * Forecast horizon = (#val cases + #test cases) of that variant.
       * Map forecasted total cycle times to val/test cases of that variant.
       * For every prefix row, compute:
            D_hat_ets_var  = predicted total cycle time (per-variant model)
            R_hat_ets_var  = max(D_hat_ets_var - elapsed_time, 0)

Outputs:
- One CSV per val/test input, with extra ETS columns, written to OUTPUT_FOLDER:
    D_hat_ets,      R_hat_ets      (global model)
    D_hat_ets_var,  R_hat_ets_var  (per-variant model, if available)
"""

import os
import glob

import numpy as np
import pandas as pd

from statsforecast import StatsForecast
from statsforecast.models import AutoETS


# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
PREFIX_INPUT_FOLDER = r"out/251110/prefix_datasets"        # where *_prefix.csv live
OUTPUT_FOLDER       = r"out/251110/prefix_with_ets"        # where we'll save predictions

# Name of the variant / product type column in the prefix files
# Change this if your column is called differently (e.g. "product_variant" or "l")
VARIANT_COLUMN      = "process"

# AutoETS config: no seasonality, automatic error/trend choice
AUTOETS_MODEL      = "ZZN"   # automatic error/trend, no seasonal component
AUTOETS_SEASON_LEN = 1       # no meaningful seasonality in case sequence
# ------------------------------------------------------------------


def load_prefix_file(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = [
        "case_id",
        "timestamp",
        "elapsed_time",
        "remaining_time",
        "case_start_time",
        "case_complete_time",
    ]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"{path} missing required column: {col}")

    return df


def build_case_table(df_prefix: pd.DataFrame, variant_col: str | None = None) -> pd.DataFrame:
    """
    From a prefix dataset, get one row per case with:
    - case_id
    - case_start_time
    - case_complete_time
    - (optional) variant
    - cycle_time = complete - start

    We take the first row per case (they all share the same start/complete times).
    """
    base_cols = ["case_id", "case_start_time", "case_complete_time"]
    cols = base_cols.copy()

    if variant_col is not None and variant_col in df_prefix.columns:
        cols.append(variant_col)

    cases = (
        df_prefix
        .sort_values(["case_id", "timestamp"])
        .drop_duplicates(subset=["case_id"], keep="first")
        [cols]
        .copy()
    )
    cases["cycle_time"] = cases["case_complete_time"] - cases["case_start_time"]
    # sort by start time to define the time-series order
    cases = cases.sort_values("case_start_time").reset_index(drop=True)
    return cases


def build_ts_df(cases: pd.DataFrame, unique_id: str) -> pd.DataFrame:
    """
    Build the statsforecast-style time series dataframe for AutoETS:

        columns: [unique_id, ds, y]

    where:
      - unique_id: same string for all rows (each scenario or variant is one series)
      - ds: integer time index (0,1,2,...)
      - y: cycle_time
    """
    n = len(cases)
    ts_df = pd.DataFrame({
        "unique_id": unique_id,
        "ds": np.arange(n),
        "y": cases["cycle_time"].to_numpy(),
    })
    return ts_df


def fit_autoets(train_ts: pd.DataFrame) -> StatsForecast:
    """
    Fit AutoETS on the training time series.
    """
    sf = StatsForecast(
        models=[AutoETS(model=AUTOETS_MODEL, season_length=AUTOETS_SEASON_LEN)],
        freq=1,   # arbitrary integer index frequency
    )
    sf.fit(df=train_ts)
    return sf


def forecast_cycle_times(sf: StatsForecast, h: int) -> np.ndarray:
    """
    Forecast h future cycle times using AutoETS.
    Returns a 1D numpy array of length h.
    """
    y_hat = sf.predict(h=h)
    # column name is the model class name, here "AutoETS"
    preds = y_hat["AutoETS"].to_numpy()
    if len(preds) != h:
        raise RuntimeError(f"Expected {h} forecasts, got {len(preds)}")
    return preds


def add_ets_predictions_to_prefix(
    df_prefix: pd.DataFrame,
    case_table: pd.DataFrame,
    D_hat: np.ndarray,
    case_ids_in_order: np.ndarray,
    col_D: str = "D_hat_ets",
    col_R: str = "R_hat_ets",
) -> pd.DataFrame:
    """
    Attach case-level ETS predictions to every prefix row.

    Inputs:
    - df_prefix: prefix-level dataframe (val or test).
    - case_table: dataframe with one row per case (not used, but kept for symmetry).
    - D_hat: array of predicted total cycle times, aligned with case_ids_in_order.
    - case_ids_in_order: np.array of case_ids (sorted) that correspond to D_hat.
    - col_D / col_R: column names for total and remaining time predictions.

    Output:
    - df_prefix copy with:
        * col_D
        * col_R = max(col_D - elapsed_time, 0)
    """
    df_prefix = df_prefix.copy()

    # map from case_id -> predicted total cycle time
    case_pred_df = pd.DataFrame({
        "case_id": case_ids_in_order,
        col_D: D_hat,
    })

    # merge predictions into prefix rows
    df_prefix = df_prefix.merge(case_pred_df, on="case_id", how="left")

    # compute predicted remaining time
    df_prefix[col_R] = np.clip(
        df_prefix[col_D] - df_prefix["elapsed_time"],
        a_min=0.0,
        a_max=None,
    )

    return df_prefix


def fit_and_predict_per_variant(
    base_root: str,
    cases_train: pd.DataFrame,
    cases_val: pd.DataFrame,
    cases_test: pd.DataFrame,
    variant_col: str,
):
    """
    Fit separate ETS models per variant and produce predictions per case.

    Returns two dicts:
        val_pred_by_case  : {case_id -> D_hat_ets_var}
        test_pred_by_case : {case_id -> D_hat_ets_var}

    If variant_col is missing or has no values in train, returns empty dicts.
    """
    if variant_col not in cases_train.columns:
        print(f"  [per-variant] Column '{variant_col}' not in cases_train -> skip per-variant ETS.")
        return {}, {}

    # Variants present in TRAIN data
    variants = sorted(cases_train[variant_col].dropna().unique())
    if len(variants) == 0:
        print(f"  [per-variant] No variants in train -> skip per-variant ETS.")
        return {}, {}

    print(f"  [per-variant] Found {len(variants)} variant(s) in TRAIN: {variants}")

    # Index val/test by case_id for easy mapping later
    val_pred_by_case = {}
    test_pred_by_case = {}

    for v in variants:
        train_v = cases_train[cases_train[variant_col] == v].copy()
        val_v   = cases_val[cases_val[variant_col] == v].copy() if variant_col in cases_val.columns else cases_val.iloc[0:0].copy()
        test_v  = cases_test[cases_test[variant_col] == v].copy() if variant_col in cases_test.columns else cases_test.iloc[0:0].copy()

        n_train_v = len(train_v)
        n_val_v   = len(val_v)
        n_test_v  = len(test_v)

        if n_train_v == 0:
            continue  # nothing to train for this variant

        print(f"    Variant {v!r}: train cases={n_train_v}, val cases={n_val_v}, test cases={n_test_v}")

        # Build per-variant training series
        unique_id = f"{base_root}_var_{v}"
        ts_train_v = build_ts_df(train_v, unique_id=unique_id)

        sf_v = fit_autoets(ts_train_v)

        h_val_v = n_val_v
        h_test_v = n_test_v
        h_total_v = h_val_v + h_test_v

        if h_total_v == 0:
            # No val/test cases for this variant -> nothing to predict
            continue

        D_hat_all_v = forecast_cycle_times(sf_v, h=h_total_v)
        D_hat_val_v = D_hat_all_v[:h_val_v]
        D_hat_test_v = D_hat_all_v[h_val_v:]

        # Map predictions back to case_ids
        if n_val_v > 0:
            case_ids_val_v = val_v["case_id"].to_numpy()
            for cid, d in zip(case_ids_val_v, D_hat_val_v):
                val_pred_by_case[cid] = d

        if n_test_v > 0:
            case_ids_test_v = test_v["case_id"].to_numpy()
            for cid, d in zip(case_ids_test_v, D_hat_test_v):
                test_pred_by_case[cid] = d

    return val_pred_by_case, test_pred_by_case


def process_scenario(train_path: str):
    """
    For one "base" scenario, find its train/val/test prefix files,
    fit global ETS + per-variant ETS (if possible), predict for val+test,
    and write out new CSVs.
    """
    base_name = os.path.basename(train_path)
    base_root = base_name.replace("_warm10_train_prefix.csv", "")
    print(f"\n=== Scenario: {base_root} ===")

    # infer paths
    val_path  = os.path.join(
        PREFIX_INPUT_FOLDER, f"{base_root}_warm10_val_prefix.csv"
    )
    test_path = os.path.join(
        PREFIX_INPUT_FOLDER, f"{base_root}_warm10_test_prefix.csv"
    )

    if not os.path.exists(val_path) or not os.path.exists(test_path):
        print(f"  Skipping: missing val or test for {base_root}")
        return

    # --- load prefix splits ---
    df_train = load_prefix_file(train_path)
    df_val   = load_prefix_file(val_path)
    df_test  = load_prefix_file(test_path)

    print(f"  Train prefixes: {len(df_train)}, cases: {df_train['case_id'].nunique()}")
    print(f"  Val   prefixes: {len(df_val)}, cases: {df_val['case_id'].nunique()}")
    print(f"  Test  prefixes: {len(df_test)}, cases: {df_test['case_id'].nunique()}")

    # --- build case tables (one row per case) ---
    cases_train = build_case_table(df_train, variant_col=VARIANT_COLUMN)
    cases_val   = build_case_table(df_val,   variant_col=VARIANT_COLUMN)
    cases_test  = build_case_table(df_test,  variant_col=VARIANT_COLUMN)

    print(f"  Train cases: {len(cases_train)}, "
          f"Val cases: {len(cases_val)}, Test cases: {len(cases_test)}")

    if len(cases_train) == 0:
        print("  No train cases -> skip.")
        return

    # --- GLOBAL ETS: build training time series for statsforecast ---
    unique_id = base_root  # any string to identify this series
    ts_train  = build_ts_df(cases_train, unique_id=unique_id)

    # --- fit AutoETS on global train series ---
    sf_global = fit_autoets(ts_train)

    # --- forecast cycle times for val + test cases (global) ---
    h_val  = len(cases_val)
    h_test = len(cases_test)
    h_total = h_val + h_test

    if h_total == 0:
        print("  No val/test cases -> nothing to predict.")
        return

    D_hat_all = forecast_cycle_times(sf_global, h=h_total)
    D_hat_val = D_hat_all[:h_val]
    D_hat_test = D_hat_all[h_val:]

    # case_ids in the same order as cases_val / cases_test
    case_ids_val  = cases_val["case_id"].to_numpy()
    case_ids_test = cases_test["case_id"].to_numpy()

    # --- attach GLOBAL predictions to prefix rows ---
    df_val_ets = add_ets_predictions_to_prefix(
        df_val, cases_val, D_hat_val, case_ids_val,
        col_D="D_hat_ets", col_R="R_hat_ets"
    )
    df_test_ets = add_ets_predictions_to_prefix(
        df_test, cases_test, D_hat_test, case_ids_test,
        col_D="D_hat_ets", col_R="R_hat_ets"
    )

    # --- PER-VARIANT ETS: optional, only if VARIANT_COLUMN is present ---
    if VARIANT_COLUMN in cases_train.columns:
        val_pred_by_case, test_pred_by_case = fit_and_predict_per_variant(
            base_root,
            cases_train,
            cases_val,
            cases_test,
            variant_col=VARIANT_COLUMN,
        )

        # Build mapping DataFrames and merge
        if val_pred_by_case:
            case_pred_val_df = pd.DataFrame({
                "case_id": list(val_pred_by_case.keys()),
                "D_hat_ets_var": list(val_pred_by_case.values()),
            })
            df_val_ets = df_val_ets.merge(case_pred_val_df, on="case_id", how="left")
            df_val_ets["R_hat_ets_var"] = np.clip(
                df_val_ets["D_hat_ets_var"] - df_val_ets["elapsed_time"],
                a_min=0.0,
                a_max=None,
            )
        else:
            print("  [per-variant] No val predictions produced (maybe no val cases with train variants).")

        if test_pred_by_case:
            case_pred_test_df = pd.DataFrame({
                "case_id": list(test_pred_by_case.keys()),
                "D_hat_ets_var": list(test_pred_by_case.values()),
            })
            df_test_ets = df_test_ets.merge(case_pred_test_df, on="case_id", how="left")
            df_test_ets["R_hat_ets_var"] = np.clip(
                df_test_ets["D_hat_ets_var"] - df_test_ets["elapsed_time"],
                a_min=0.0,
                a_max=None,
            )
        else:
            print("  [per-variant] No test predictions produced (maybe no test cases with train variants).")
    else:
        print(f"  [per-variant] Column '{VARIANT_COLUMN}' not found in case tables -> skipping per-variant ETS.")

    # --- save outputs ---
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    out_val  = os.path.join(OUTPUT_FOLDER, f"{base_root}_warm10_val_prefix_ets.csv")
    out_test = os.path.join(OUTPUT_FOLDER, f"{base_root}_warm10_test_prefix_ets.csv")

    df_val_ets.to_csv(out_val, index=False)
    df_test_ets.to_csv(out_test, index=False)

    print(f"  Saved:\n"
          f"    {out_val}  ({len(df_val_ets)} rows)\n"
          f"    {out_test} ({len(df_test_ets)} rows)")


def main():
    if not os.path.isdir(PREFIX_INPUT_FOLDER):
        raise SystemExit(f"Prefix folder not found: {PREFIX_INPUT_FOLDER}")

    # find all *_warm10_train_prefix.csv as "base" scenarios
    train_files = sorted(
        glob.glob(os.path.join(PREFIX_INPUT_FOLDER, "*_warm10_train_prefix.csv"))
    )

    if not train_files:
        raise SystemExit(f"No '*_warm10_train_prefix.csv' files in {PREFIX_INPUT_FOLDER}")

    print(f"Found {len(train_files)} scenario(s) to process.")
    for train_path in train_files:
        process_scenario(train_path)


if __name__ == "__main__":
    main()


Found 8 scenario(s) to process.

=== Scenario: log_FIFO_run0_EXP_dedicated_C1 ===
  Train prefixes: 142404, cases: 4491
  Val   prefixes: 47885, cases: 1508
  Test  prefixes: 47885, cases: 1508
  Train cases: 4491, Val cases: 1508, Test cases: 1508
  [per-variant] Found 18 variant(s) in TRAIN: ['variant_1', 'variant_10', 'variant_11', 'variant_12', 'variant_13', 'variant_14', 'variant_15', 'variant_16', 'variant_17', 'variant_18', 'variant_2', 'variant_3', 'variant_4', 'variant_5', 'variant_6', 'variant_7', 'variant_8', 'variant_9']
    Variant 'variant_1': train cases=249, val cases=64, test cases=80
    Variant 'variant_10': train cases=274, val cases=76, test cases=85
    Variant 'variant_11': train cases=227, val cases=88, test cases=65
    Variant 'variant_12': train cases=259, val cases=80, test cases=81
    Variant 'variant_13': train cases=241, val cases=81, test cases=90
    Variant 'variant_14': train cases=266, val cases=79, test cases=84
    Variant 'variant_15': train case

In [ ]:
#!/usr/bin/env python

"""
Compute MAE and MAPE for ETS remaining-time predictions on the *validation* prefixes.

Assumes:
- You already ran the ETS script.
- In out/251110/prefix_with_ets you have files like:
    ..._warm10_val_prefix_ets.csv
- Each file has columns:
    remaining_time   (true)
    R_hat_ets        (predicted by global ETS)
  and optionally:
    R_hat_ets_var    (predicted by per-variant ETS)

MAPE is computed only on rows with remaining_time != 0
(to avoid division by zero).
"""

import os
import glob

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# CONFIG: change if your folder is different
# ------------------------------------------------------------------
ETS_FOLDER = r"out/251110/prefix_with_ets"
VAL_PATTERN = "*_warm10_val_prefix_ets.csv"
# ------------------------------------------------------------------


def _mae_mape(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float]:
    """Helper: compute MAE and MAPE (ignoring y_true == 0 for MAPE)."""
    mae = np.mean(np.abs(y_true - y_pred))

    mask = y_true != 0
    if mask.any():
        mape = np.mean(
            np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
        ) * 100.0
    else:
        mape = np.nan

    return mae, mape


def compute_metrics_for_file(path: str):
    """
    Load one ETS val file and compute:
      - MAE/MAPE for global ETS   (R_hat_ets)
      - MAE/MAPE for per-variant  (R_hat_ets_var), if available
    """
    df = pd.read_csv(path)

    if "remaining_time" not in df.columns:
        raise ValueError(f"{path} is missing required column: remaining_time")

    y_true = df["remaining_time"].to_numpy()

    # --- Global ETS metrics ---
    if "R_hat_ets" not in df.columns:
        raise ValueError(f"{path} is missing required column: R_hat_ets")

    y_pred_global = df["R_hat_ets"].to_numpy()
    mae_global, mape_global = _mae_mape(y_true, y_pred_global)

    # --- Per-variant ETS metrics (optional) ---
    if "R_hat_ets_var" in df.columns:
        y_pred_var = df["R_hat_ets_var"].to_numpy()
        mae_var, mape_var = _mae_mape(y_true, y_pred_var)
    else:
        mae_var, mape_var = np.nan, np.nan

    return mae_global, mape_global, mae_var, mape_var


def main():
    if not os.path.isdir(ETS_FOLDER):
        raise SystemExit(f"ETS folder not found: {ETS_FOLDER}")

    pattern = os.path.join(ETS_FOLDER, VAL_PATTERN)
    val_files = sorted(glob.glob(pattern))

    if not val_files:
        raise SystemExit(f"No '{VAL_PATTERN}' files found in {ETS_FOLDER}")

    print(f"Found {len(val_files)} validation file(s).")

    maes_g, mapes_g = [], []
    maes_v, mapes_v = [], []

    for path in val_files:
        mae_g, mape_g, mae_v, mape_v = compute_metrics_for_file(path)
        maes_g.append(mae_g)
        mapes_g.append(mape_g)
        maes_v.append(mae_v)
        mapes_v.append(mape_v)

        print(
            f"{os.path.basename(path)}  ->  "
            f"GLOBAL: MAE = {mae_g:.4f}, MAPE = {mape_g:.2f}%   "
            f"PER-VARIANT: MAE = {mae_v:.4f}, MAPE = {mape_v:.2f}%"
        )

    overall_mae_g = float(np.mean(maes_g))
    overall_mape_g = float(np.nanmean(mapes_g))  # ignore NaNs if any

    # per-variant may be NaN if some files don't have per-variant predictions
    overall_mae_v = float(np.nanmean(maes_v))
    overall_mape_v = float(np.nanmean(mapes_v))

    print("\n------------------------------")
    print(f"GLOBAL ETS over all val files:")
    print(f"  Average MAE   : {overall_mae_g:.4f}")
    print(f"  Average MAPE  : {overall_mape_g:.2f}%")

    print("\nPER-VARIANT ETS over all val files (ignoring NaNs):")
    print(f"  Average MAE   : {overall_mae_v:.4f}")
    print(f"  Average MAPE  : {overall_mape_v:.2f}%")


if __name__ == "__main__":
    main()


Found 8 validation file(s).
log_FIFO_run0_EXP_dedicated_C1_warm10_val_prefix_ets.csv  ->  MAE = 19.6304,  MAPE = 1562.97%
log_FIFO_run0_EXP_hybrid30_C1_warm10_val_prefix_ets.csv  ->  MAE = 16.4547,  MAPE = 244.94%
log_FIFO_run0_EXP_pooled_C1_warm10_val_prefix_ets.csv  ->  MAE = 20.4916,  MAPE = 332.63%
log_FIFO_run0_actuator_manufacturing_no_rework_warm10_val_prefix_ets.csv  ->  MAE = 24.1462,  MAPE = 135.56%
log_FIFO_run0_actuator_manufacturing_with_rework_warm10_val_prefix_ets.csv  ->  MAE = 25.1946,  MAPE = 158.36%
log_FIFO_run0_actuator_mfg_pooledM_dedicatedA1_no_rework_warm10_val_prefix_ets.csv  ->  MAE = 18.1238,  MAPE = 176.21%
log_FIFO_run0_actuator_mfg_pooledM_dedicatedA1_with_rework_warm10_val_prefix_ets.csv  ->  MAE = 39.7816,  MAPE = 139.84%
log_FIFO_run0_all_dedicated_sticky_with_rework_warm10_val_prefix_ets.csv  ->  MAE = 33.3266,  MAPE = 906.02%

------------------------------
Average MAE over all val files:   24.6437
Average MAPE over all val files:  457.07%


In [1]:
%pip install statsforecast


Note: you may need to restart the kernel to use updated packages.
